# Github Repository Chroma Query

In [19]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
from collections import Counter
from datetime import datetime, timezone


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from apps.agentic.core.document_loaders.github_document_loader import GitHubChromaDocumentLoader
from apps.agentic.core.constants import GITHUB_DB_NAME, GITHUB_COLLECTION_NAME
from apps.agentic.core.utils import (set_chatgpt_env, set_langsmith_env, set_tavily_env, 
                                     set_github_env)

DB_PATH = str(Path("../../.db").resolve())

set_chatgpt_env(filedir="../../.keys")
set_langsmith_env(filedir="../../.keys")

## Verify Document Counts

In [20]:
GITHUB_DB_NAME, GITHUB_COLLECTION_NAME, DB_PATH, os.listdir(DB_PATH)

('github',
 'github-repos',
 '/Users/troy/Develop/gly.fish/yada/.db',
 ['.DS_Store', 'research_library', 'fred', 'github', 'etf'])

In [21]:
vs = GitHubChromaDocumentLoader(DB_PATH).vectorstore
coll = vs._collection

In [22]:
client = vs._client
print([c.name for c in client.list_collections()])
print("Opened:", coll.name)

['github-repos']
Opened: github-repos


In [23]:
print("total docs:", coll.count())

total docs: 415006


In [24]:
res = coll.get(where={"repo": "zgomot"})   # no 'include' arg at all
print("zgomot docs:", len(res["ids"]))

zgomot docs: 384


## Verify Document Metadata

In [ ]:
# A sample, only to enumerate which metadata keys exist -- NOT a full scan.
# The cells under "Metadata" below rebuild `metas` over the whole collection.
probe = coll.get(limit=10000, include=["metadatas"])  # adjust as needed
metas = probe.get("metadatas", [])
if metas:
    metas  = [m for m in metas if m]
keys = set().union(*(m.keys() for m in metas)) if metas else set()
for key in sorted(keys):
    print(key)

In [26]:
if metas:
    print("repo counts:", Counter(m.get("repo") for m in metas if "repo" in m and m.get("repo")).most_common(10))
    print("account counts:", Counter(m.get("account") for m in metas if "account" in m and m.get("account")).most_common(10))

repo counts: [('autobahn-python', 2937), ('BlueCap', 2809), ('has_ancestor', 2726), ('cpp-ethereum', 1199), ('zgomot', 192), ('acts_as_aln', 78), ('PracticeSessions', 43), ('data_sink', 16)]
account counts: [('troystribling', 10000)]


## Search Validation

### Similarity

In [27]:
vs = GitHubChromaDocumentLoader(DB_PATH).vectorstore

docs = vs.similarity_search("README summary", k=5, filter={"repo": "zgomot"})
for d in docs:
    print(d.metadata.get("path"), "—", d.metadata.get("commit_ts"))
    print(d.page_content[:160].replace("\n"," "), "\n")

LICENSE — 2009-08-08T02:33:39+00:00
The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.  THE SOFTWARE IS PROVIDED "AS IS" 

LICENSE — 2009-08-08T02:33:39+00:00
The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.  THE SOFTWARE IS PROVIDED "AS IS" 

LICENSE — 2009-08-08T02:33:39+00:00
The MIT License  Copyright (c) 2009 troystribling  Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated  

LICENSE — 2009-08-08T02:33:39+00:00
The MIT License  Copyright (c) 2009 troystribling  Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated  

VERSION — 2015-03-07T19:32:16+00:00
1.1.0 



### MMR Search

In [28]:
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8, "fetch_k": 50, "filter": {"repo": "zgomot"}}
)

hits = retriever.invoke("Where is MIDI output handled?")
for d in hits[:5]:
    print(d.metadata.get("path"), d.metadata.get("commit_ts"))

lib/zgomot/drivers/core_midi.rb 2013-08-17T22:04:27+00:00
lib/zgomot/midi/note.rb 2013-08-17T02:36:11+00:00
lib/zgomot/ui/windows.rb 2013-08-14T04:55:40+00:00
README.rdoc 2015-03-07T20:58:50+00:00
README.rdoc 2015-03-07T20:58:50+00:00


## Search Examples 

In [29]:
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
        "filter": {"$and": [{"repo": "zgomot"}, {"ext": ".rb"}, {"account": "troystribling"}]},
    },
)
hits = retriever.invoke("Where is MIDI output handled? (drivers, note_on, note_off, port, send)")

[print(h) for h in hits]

page_content='class Zgomot::Drivers

  class CoreMidi < Driver

    attr_reader :destinations, :sources, :input, :output

    # API Wrapper
    module Interface

      extend FFI::Library
      ffi_lib '/System/Library/Frameworks/CoreMIDI.framework/Versions/Current/CoreMIDI'

      typedef :pointer, :CFStringRef
      typedef :int32,   :ItemCount
      typedef :pointer, :MIDIClientRef
      typedef :pointer, :MIDIDeviceRef
      typedef :pointer, :MIDIEntityRef
      typedef :pointer, :MIDIObjectRef
      typedef :pointer, :MIDIEndpointRef
      typedef :uint32,  :MIDITimeStamp
      typedef :pointer, :MIDIPortRef
      typedef :int32,   :OSStatus

      class MIDIPacket < FFI::Struct
        layout :timestamp,  :MIDITimeStamp,
               :nothing,    :uint32,
               :length,     :uint16,
               :data,       [:uint8, 256]
      end

      class MIDIPacketList < FFI::Struct
        layout :numPackets,   :uint32,
               :packet,       [MIDIPacket.by_value, 1]


[None, None, None, None, None, None, None, None]

### Use MMR Call Directly

In [30]:
docs = vs.max_marginal_relevance_search(
    "Where is MIDI output handled? (drivers, note_on, note_off, port, send)",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
        "filter": {"$and": [{"repo": "zgomot"}, {"ext": ".rb"}]},
    },
)

[print(h) for h in docs]

page_content='udd96x6snlR48e1bx58zRq1Cjdfffdatasmd544w233xhg9JRIa5+gb4azz+H+dh6YM2eO7r33Xi1btszdocCPcA7IxvHn+BspjM479j7lmfEbzhYcHKznnntO/fv3t7g8IyND7733noujup6vjd25Jdhd9WYhAIB386fcr6+zNsfmyvxfXnwhj26NokWLWr2OI98KAvv5eh4QuMIb5vmS7831HcnfckVwH38/Dv19/33pfgGJewY8oW9bkXOCq/na+E/OCQDsR8EnPEZgYKCqVaum+++/X6NHj9bvv/+udevW6ZNPPlGfPn1Urlw5u/pfs2aNgyK1zcKFCzVr1iyH9efJT7cwmqTe6Pbbb8/1le3X+ueff657ik1ub/eU/iv4vPHfb7Rjx47rfriuXbvWVDyS+97ueUVAQICGDBliqu3EiROd8p2xpU97bly1dl2z30VrREVFObxPaxUoUMB0W2svhpg9Jl3hs88+0xtvvOGyRKYny+3vxRnfc8CdcrtwdObMGYduy6g/T03e+MP+X3Hvvffqp59+yvV3XG7i4+P13Xff6YEHHlCnTp30xx9/uOUmL6NzmDPOt9b8PkDu5syZo6FDh3r0nAu+iXMAx98Vjk7yG/Xna/+f+f74rsGDB6tWrVoWl23YsEFbt251cUT/8bWxO7drDJ50zQgA4B18Pffr66z9XW3PtSlHP/TGk+cEzsjp2PL/zxNvpvZn5AHhbzx5ni/53lwf8Eb+fhz6+/5LvnO/gMQ9A96KnBPcwdfGf3JOAGA/6177BbhYoUKF1KZNG7Vp00YjRozQpk2b9PXXX9v0xNZ//vnHCRGal5WVpZEjRyooKEidO3e2uz9PTsJkZmaaahcaGqq6detq1apVeba9fPmytmzZonr16knK+WbOG/utXbv21f+uX7++5s6da7Ftenq6/v77b7Vo0SLPfm/k7oJPSWrWrJkaN26s2NjYXNsdP35c06ZNM/VG

[None, None, None, None]

In [31]:
after = datetime(2014,1,1, tzinfo=timezone.utc).timestamp()

flt = {
  "$and": [
    {"repo": "zgomot"},
    {"ext": ".rb"},
    {"commit_ts": {"$gte": after}}
  ]
}
docs = vs.similarity_search("note_on handler", k=8, filter=flt)
[print(h) for h in hits]

page_content='class Zgomot::Drivers

  class CoreMidi < Driver

    attr_reader :destinations, :sources, :input, :output

    # API Wrapper
    module Interface

      extend FFI::Library
      ffi_lib '/System/Library/Frameworks/CoreMIDI.framework/Versions/Current/CoreMIDI'

      typedef :pointer, :CFStringRef
      typedef :int32,   :ItemCount
      typedef :pointer, :MIDIClientRef
      typedef :pointer, :MIDIDeviceRef
      typedef :pointer, :MIDIEntityRef
      typedef :pointer, :MIDIObjectRef
      typedef :pointer, :MIDIEndpointRef
      typedef :uint32,  :MIDITimeStamp
      typedef :pointer, :MIDIPortRef
      typedef :int32,   :OSStatus

      class MIDIPacket < FFI::Struct
        layout :timestamp,  :MIDITimeStamp,
               :nothing,    :uint32,
               :length,     :uint16,
               :data,       [:uint8, 256]
      end

      class MIDIPacketList < FFI::Struct
        layout :numPackets,   :uint32,
               :packet,       [MIDIPacket.by_value, 1]


[None, None, None, None, None, None, None, None]

### Metadata

In [33]:
vs = GitHubChromaDocumentLoader(DB_PATH).vectorstore
collection = vs._collection

# collection.get() binds one SQL variable per returned row, and SQLite caps at
# 32766 -- so a single get() over all 415k chunks raises "too many SQL variables".
# Page instead, as etf_document_loader does. include=["metadatas"] also keeps the
# ~375 MB of page_content out of the result; this cell only needs metadata.
BATCH = 20_000
total = collection.count()

metas = []
for offset in range(0, total, BATCH):
    result = collection.get(limit=BATCH, offset=offset, include=["metadatas"])
    metas.extend(m for m in result.get("metadatas") or [] if m)

len(metas)

415006

In [34]:
repos =  Counter(m.get("repo") for m in metas if "repo" in m and m.get("repo")).most_common(200)
print("Repositories", len(repos))
for repo in repos:
    print(repo)

Repositories 101
('alef', 106553)
('BAP3', 73029)
('Studium', 34540)
('pyalgotrade', 32106)
('website', 23142)
('alpaca', 19063)
('meida', 15000)
('zipline', 9588)
('cryptocoins', 8935)
('backtrader', 8543)
('libetpan', 7054)
('ejabberd', 6634)
('PyMC', 6355)
('Financial-Models-Numerical-Methods', 6187)
('seeker1', 4193)
('autobahn-python', 2937)
('streethak', 2901)
('acts_as_aln', 2898)
('planb_site', 2864)
('route-me', 2818)
('BlueCap', 2809)
('has_ancestor', 2726)
('iOSPorts', 2646)
('nordic-ble-sdk-examples', 2493)
('truffle_examples', 2252)
('GPUImage', 2231)
('arduino-examples', 2007)
('MailCore', 1584)
('gnosus-xmpp', 1377)
('aethyr', 1311)
('cpp-ethereum', 1199)
('aethyr_agent', 1004)
('photio', 996)
('BlueCap-ObjC', 949)
('hue-switch-arduino', 913)
('NN', 776)
('agentic', 772)
('navi', 703)
('pylivetrader', 641)
('agent_xmpp', 604)
('NestPWN', 604)
('LRResty', 546)
('FutureLocation', 509)
('exempli', 476)
('ep-chan-book-algo-trading', 437)
('shade', 434)
('MCMC', 412)
('zgomot

## Metadata Key Values

Scan the full collection to find all distinct values for each filterable metadata field.
Use these to validate filter values before passing them to the agent or similarity search.

In [35]:
# `metas` is the full-collection scan loaded above; reuse it here.
FILTER_KEYS = ["account", "repo", "ext"]

distinct: dict[str, set] = {k: set() for k in FILTER_KEYS}
for meta in metas:
    for key in FILTER_KEYS:
        val = meta.get(key)
        if val:
            distinct[key].add(val)

for key in FILTER_KEYS:
    values = sorted(distinct[key])
    print(f"\n{key} ({len(values)} distinct values):")
    for v in values:
        print(f"  {v!r}")


account (2 distinct values):
  'glyfish'
  'troystribling'

repo (101 distinct values):
  'BAP3'
  'BLEBond'
  'BlueBeacon'
  'BlueCap'
  'BlueCap-ObjC'
  'CBPeripheralManager-Demo'
  'DetectPencil'
  'Financial-Models-Numerical-Methods'
  'FutureLocation'
  'GPUImage'
  'HueSwitch-ObjC'
  'LRResty'
  'LocationTest'
  'MCMC'
  'MailCore'
  'MemoryMappedFileObjC'
  'MemoryMappedFileSwift'
  'NN'
  'NestPWN'
  'PID'
  'PracticeSessions'
  'Principia'
  'PyMC'
  'Robinhood'
  'RobinhoodShell'
  'Rpi_StartKit'
  'Rpi_StartKit_Swift'
  'Sano'
  'SimpleFutures'
  'Stochastic.jl'
  'Studium'
  'Studium-ObjC'
  'Swift-Errors'
  'WaitTime'
  'acts_as_aln'
  'aethyr'
  'aethyr_agent'
  'agent_linux'
  'agent_xmpp'
  'agent_xmpp_orig'
  'agentic'
  'alef'
  'alpaca'
  'alpaca-backtrader-api'
  'alpaca-trade-api-python'
  'arduino-examples'
  'autobahn-python'
  'backtrader'
  'bluecap-arduino'
  'bluecap-arduino-examples'
  'cpp-ethereum'
  'cryptocoins'
  'data_examples'
  'data_sink'
  'dotfil

In [36]:
# Search within a key's distinct values to validate a specific term
search_key = "ext"
search_term = ".py"

matches = sorted(v for v in distinct[search_key] if search_term.lower() in v.lower())
print(f"{search_key} values containing {search_term!r}:")
for v in matches:
    print(f"  {v!r}")

ext values containing '.py':
  '.py'
  '.pyi'
  '.pyx'


In [37]:
zgomot_metas = [m for m in metas if "repo" in m and m.get("repo") == "zgomot"]
Counter(m.get("path") for m in zgomot_metas if "path" in m and m.get("path"))

Counter({'README.rdoc': 66,
         'lib/zgomot/ui/windows.rb': 38,
         'lib/zgomot/drivers/core_midi.rb': 20,
         'lib/zgomot/midi/stream.rb': 16,
         'lib/zgomot/midi/clock.rb': 14,
         'zgomot.gemspec': 12,
         'lib/zgomot/comp/chord.rb': 12,
         'lib/zgomot/main.rb': 10,
         'lib/zgomot/comp/progression.rb': 10,
         'lib/zgomot/midi/cc.rb': 10,
         'lib/zgomot/comp/note_list.rb': 8,
         'lib/zgomot/comp/perc.rb': 8,
         'lib/zgomot/midi/note.rb': 8,
         'lib/zgomot/ui/output.rb': 8,
         'LICENSE': 4,
         'Rakefile': 4,
         'lib/zgomot/comp/mode.rb': 4,
         'lib/zgomot/comp/pattern.rb': 4,
         'lib/zgomot/drivers/driver.rb': 4,
         'lib/zgomot/drivers/mgr.rb': 4,
         'lib/zgomot/midi/channel.rb': 4,
         'lib/zgomot/midi/dispatcher.rb': 4,
         '.document': 2,
         '.gitignore': 2,
         '.rvmrc': 2,
         'VERSION': 2,
         'zgomot.gems': 2,
         'bin/zgomot': 2